# Aviation Accidents Analysis

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident. They are also interested in any general variables/conditions that might be at play. Your analysis will be based off of aviation accident data accumulated from the years 1948-2023. 

Our client is only interested in airplane makes/models that are professional builds and could potentially still be active. Assume a max lifetime of 40 years for a make/model retirement and make sure to filter your data accordingly (i.e. from 1983 onwards). They would also like separate recommendations for small aircraft vs. larger passenger models. **In addition, make sure that claims that you make are statistically robust and that you have enough samples when making comparisons between groups.**


In this summative assessment you will demonstrate your ability to:
- **Use Pandas to load, inspect, and clean the dataset appropriately.**
- **Transform relevant columns to create measures that address the problem at hand.**
- conduct EDA: visualization and statistical measures to systematically understand the structure of the data
- recommend a set of airplanes and makes conforming to the client's request and identify at least *two* factors contributing to airplane safety. You must provide supporting evidence (visuals, summary statistics, tables) for each claim you make.

### Make relevant library imports

In [279]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re

## Data Loading and Inspection

### Load in data from the relevant directory and inspect the dataframe.
- inspect NaNs, datatypes, and summary statistics

In [280]:
#loading in the data

aviation_df = pd.read_csv("data/AviationData.csv", encoding="Latin-1", low_memory=False)

#changed the encoding to latin-1 as utf-8 was causing errors
#set the low_memory to false so that the entire dataset is loaded into memory

#inspecting Nans and datatypes
print(aviation_df.info())


<class 'pandas.DataFrame'>
RangeIndex: 88889 entries, 0 to 88888
Data columns (total 31 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Event.Id                88889 non-null  str    
 1   Investigation.Type      88889 non-null  str    
 2   Accident.Number         88889 non-null  str    
 3   Event.Date              88889 non-null  str    
 4   Location                88837 non-null  str    
 5   Country                 88663 non-null  str    
 6   Latitude                34382 non-null  str    
 7   Longitude               34373 non-null  str    
 8   Airport.Code            50132 non-null  str    
 9   Airport.Name            52704 non-null  str    
 10  Injury.Severity         87889 non-null  str    
 11  Aircraft.damage         85695 non-null  str    
 12  Aircraft.Category       32287 non-null  str    
 13  Registration.Number     87507 non-null  str    
 14  Make                    88826 non-null  str    
 

In [281]:
#summary statistics
#Print out summary statistics
print(aviation_df.describe())

       Number.of.Engines  Total.Fatal.Injuries  Total.Serious.Injuries  \
count       82805.000000          77488.000000            76379.000000   
mean            1.146585              0.647855                0.279881   
std             0.446510              5.485960                1.544084   
min             0.000000              0.000000                0.000000   
25%             1.000000              0.000000                0.000000   
50%             1.000000              0.000000                0.000000   
75%             1.000000              0.000000                0.000000   
max             8.000000            349.000000              161.000000   

       Total.Minor.Injuries  Total.Uninjured  
count          76956.000000     82977.000000  
mean               0.357061         5.325440  
std                2.235625        27.913634  
min                0.000000         0.000000  
25%                0.000000         0.000000  
50%                0.000000         1.000000  
75% 

In [282]:
#checking for duplicate rows
print(aviation_df.duplicated().any())

False


## Data Cleaning

### Filtering aircrafts and events

We want to filter the dataset to include aircraft that the client is interested in an analysis of:
- inspect relevant columns
- figure out any reasonable imputations
- filter the dataset

In [283]:

#printing ou the columns
print("columns:\n", aviation_df.columns)


#inspecting columns
aviation_df.head()



columns:
 Index(['Event.Id', 'Investigation.Type', 'Accident.Number', 'Event.Date',
       'Location', 'Country', 'Latitude', 'Longitude', 'Airport.Code',
       'Airport.Name', 'Injury.Severity', 'Aircraft.damage',
       'Aircraft.Category', 'Registration.Number', 'Make', 'Model',
       'Amateur.Built', 'Number.of.Engines', 'Engine.Type', 'FAR.Description',
       'Schedule', 'Purpose.of.flight', 'Air.carrier', 'Total.Fatal.Injuries',
       'Total.Serious.Injuries', 'Total.Minor.Injuries', 'Total.Uninjured',
       'Weather.Condition', 'Broad.phase.of.flight', 'Report.Status',
       'Publication.Date'],
      dtype='str')


,Event.Id,Investigation.Type,Accident.Number,Event.Date,Location,Country,Latitude,Longitude,Airport.Code,Airport.Name,...,Purpose.of.flight,Air.carrier,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured,Weather.Condition,Broad.phase.of.flight,Report.Status,Publication.Date
0,20001218X45444,Accident,SEA87LA080,1948-10-24,"MOOSE CREEK, ID",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,2.0,0.0,0.0,0.0,UNK,Cruise,Probable Cause,NaN
1,20001218X45447,Accident,LAX94LA336,1962-07-19,"BRIDGEPORT, CA",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,4.0,0.0,0.0,0.0,UNK,Unknown,Probable Cause,19-09-1996
2,20061025X01555,Accident,NYC07LA005,1974-08-30,"Saltville, VA",United States,36.922223,-81.878056,NaN,NaN,...,Personal,NaN,3.0,NaN,NaN,NaN,IMC,Cruise,Probable Cause,26-02-2007
3,20001218X45448,Accident,LAX96LA321,1977-06-19,"EUREKA, CA",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,2.0,0.0,0.0,0.0,IMC,Cruise,Probable Cause,12-09-2000
4,20041105X01764,Accident,CHI79FA064,1979-08-02,"Canton, OH",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,1.0,2.0,NaN,0.0,VMC,Approach,Probable Cause,16-04-1980


In [284]:
#count of null values in each column
print("\ncount of null values in each column:\n")
print(aviation_df.isnull().sum())


count of null values in each column:

Event.Id                      0
Investigation.Type            0
Accident.Number               0
Event.Date                    0
Location                     52
Country                     226
Latitude                  54507
Longitude                 54516
Airport.Code              38757
Airport.Name              36185
Injury.Severity            1000
Aircraft.damage            3194
Aircraft.Category         56602
Registration.Number        1382
Make                         63
Model                        92
Amateur.Built               102
Number.of.Engines          6084
Engine.Type                7096
FAR.Description           56866
Schedule                  76307
Purpose.of.flight          6192
Air.carrier               72241
Total.Fatal.Injuries      11401
Total.Serious.Injuries    12510
Total.Minor.Injuries      11933
Total.Uninjured            5912
Weather.Condition          4492
Broad.phase.of.flight     27165
Report.Status              6384
P

In [285]:
#Figure out any reasonable imputations


#Since the dataset has very many null values, we are not going to drop any rows as too many would be dropped and affect the quality of the remaining data set

#Numerical Injury Counts (Total.Fatal.Injuries, Total.Uninjured, etc.): Missing values in accident injury counts typically represent 0 casualties
# (i.e., no injuries reported for that category). We can fill these with 0.

#Categorical / Descriptive Fields (Weather.Condition, Broad.phase.of.flight, Aircraft.damage): Can be filled with a placeholder string like 'Unknown' or dropped depending on the analysis.

#Metadata & Identifiers (Latitude, Longitude, Airport.Code, Air.carrier): These have heavy missingness because many incidents occur off-airport or lack tracking data; they should be left as NaN or filtered when analyzing location-specific data.

In [286]:
#Filter the dataset
#Aircraft.Category

#Client is only interested in airplanes


airplanes_df = aviation_df[aviation_df["Aircraft.Category"] == "Airplane"]
print(airplanes_df["Aircraft.Category"])



5        Airplane
7        Airplane
8        Airplane
12       Airplane
13       Airplane
           ...   
88869    Airplane
88873    Airplane
88876    Airplane
88877    Airplane
88886    Airplane
Name: Aircraft.Category, Length: 27617, dtype: str


In [287]:
#Amateur.Built

#Client is only interested in professional builds#

airplanes_df["Amateur.Built"].unique()
pro_built_df =airplanes_df[airplanes_df["Amateur.Built"] == "No"]
pro_built_df["Amateur.Built"].unique()

<ArrowStringArray>
['No']
Length: 1, dtype: str

In [288]:
#Event.Date

#Removing all events older than 40 years ago


# 2. Convert Event.Date to datetime objects
pro_built_df["Event.Date"] = pd.to_datetime(pro_built_df["Event.Date"], errors="coerce")

# 3. Filter out events older than 40 years (cutoff: October 11, 1986)
cutoff_date = pd.to_datetime('1986-10-11')
df_filtered = pro_built_df[pro_built_df["Event.Date"] >= cutoff_date].copy()

df_filtered["Event.Date"]

16646   1987-01-15
16647   1987-01-15
16837   1987-02-14
17151   1987-04-02
19663   1988-02-16
           ...    
88869   2022-12-13
88873   2022-12-14
88876   2022-12-15
88877   2022-12-16
88886   2022-12-26
Name: Event.Date, Length: 21416, dtype: datetime64[us]

### Cleaning and constructing Key Measurables

Injuries and robustness to destruction are a key interest point for the client. Clean and impute relevant columns and then create derived fields that best quantifies what the client wishes to track. **Use commenting or markdown to explain any cleaning assumptions as well as any derived columns you create.**

**Construct metric for fatal/serious injuries**

*Hint:* Estimate the total number of passengers on each flight. The likelihood of serious / fatal injury can be estimated as a fraction from this.

In [289]:

# 1. Injury Cleaning & Derived Metrics
injury_cols = ["Total.Fatal.Injuries", "Total.Serious.Injuries", "Total.Minor.Injuries", "Total.Uninjured"]
for col in injury_cols:
    df_filtered[col] = pd.to_numeric(df_filtered[col], errors="coerce").fillna(0)

# Calculate Total Occupants
df_filtered["Total.Occupants"] = df_filtered[injury_cols].sum(axis=1)

# Calculate Severe Injury Rate (Fatal + Serious / Total Occupants)
df_filtered["Severe.Injury.Rate"] = (df_filtered["Total.Fatal.Injuries"] + df_filtered["Total.Serious.Injuries"]) / df_filtered["Total.Occupants"]
# Handle division by zero / zero occupants
df_filtered.loc[df_filtered["Total.Occupants"] == 0, "Severe.Injury.Rate"] = pd.NA



**Aircraft.Damage**
- identify and execute any cleaning tasks
- construct a derived column tracking whether an aircraft was destroyed or not.

In [290]:
# 2. Aircraft Damage Cleaning & Derived 'Is.Destroyed' Column
df_filtered["Aircraft.damage"] = df_filtered["Aircraft.damage"].astype(str).str.strip().str.capitalize()

# Create binary derived column tracking whether the aircraft was destroyed
df_filtered["Is.Destroyed"] = df_filtered["Aircraft.damage"].apply(
    lambda x: 1 if x == "Destroyed" else (0 if x in ["Substantial", "Minor"] else pd.NA)
)

print(df_filtered[["Aircraft.damage", "Is.Destroyed", "Total.Occupants", "Severe.Injury.Rate"]].head(10))

      Aircraft.damage Is.Destroyed  Total.Occupants  Severe.Injury.Rate
16646       Destroyed            1             10.0            1.000000
16647       Destroyed            1             10.0            1.000000
16837             NaN         <NA>            135.0            0.007407
17151     Substantial            0              1.0            0.000000
19663       Destroyed            1              3.0            0.000000
19931       Destroyed            1              1.0            1.000000
20014     Substantial            0              5.0            0.000000
20084             NaN         <NA>            193.0            0.005181
20200           Minor            0            258.0            0.000000
20693       Destroyed            1              1.0            1.000000


### Investigate the *Make* column
- Identify cleaning tasks here
- List cleaning tasks clearly in markdown
- Execute the cleaning tasks
- For your analysis, keep Makes with a reasonable number (you can put the threshold at 50 though lower could work as well)

In [291]:
#Identifying how many airplanes each manufacturer has
df_filtered["Make"].value_counts()





Make
CESSNA               4867
PIPER                2803
Cessna               2269
Piper                1182
BOEING               1037
                     ... 
BELL                    1
DETRICK DONALD G        1
PHANTOM                 1
JAMES R DERNOVSEK       1
ORLICAN S R O           1
Name: count, Length: 1332, dtype: int64

In [292]:
#Identifying how many makes there are
#print("\n\n")

df_filtered["Make"].unique()

<ArrowStringArray>
[               'Mooney',            'Swearingen',                'Boeing',
                 'Piper',            'Gulfstream',                 'Beech',
                 'Helio',                'Cessna',      'Airbus Industrie',
               'Univair',
 ...
         'SCOTT TERRY G',              'BEARHAWK', 'ECLIPSE AEROSPACE INC',
       'PIPISTREL D O O',   'MULHOLLAND ROBERT A',                  'BELL',
      'DETRICK DONALD G',               'PHANTOM',     'JAMES R DERNOVSEK',
         'ORLICAN S R O']
Length: 1333, dtype: str

# Cleaning Tasks
- Standardize Casing. e.g CESSNA and Cessna appear as different makes because of the case inconsistency
- Remove whitespaces both before and after the string using strip

In [293]:
# Cleaning function for Make
def clean_make(val):
    if pd.isna(val):
        return pd.NA
    s = str(val).strip()
    # Remove leading/trailing quotes or punctuation
    s = re.sub(r'^[\"\'\-\_\*\(\)]+|[\"\'\-\_\*\(\)]+$', '', s)
    # Standardize internal whitespace
    s = re.sub(r'\s+', ' ', s)
    # Standardize casing to Title Case
    return s.title()

# Apply cleaning
df_filtered["Clean.Make"] = df_filtered["Make"].apply(clean_make)

# Filter for makes with a frequency >= 50
make_counts = df_filtered["Clean.Make"].value_counts()
valid_makes = make_counts[make_counts >= 50].index
df_cleaned_makes = df_filtered[df_filtered["Clean.Make"].isin(valid_makes)].copy()

print(f"Original unique makes: {df_filtered['Make'].nunique()}")
print(f"Unique makes after cleaning: {df_filtered['Clean.Make'].nunique()}")
print(f"Unique makes retained (>= 50 occurrences): {len(valid_makes)}")
print(f"Filtered dataset row count: {len(df_cleaned_makes)}")

Original unique makes: 1332
Unique makes after cleaning: 1086
Unique makes retained (>= 50 occurrences): 36
Filtered dataset row count: 17870


### Inspect Model column
- Get rid of any NaNs.
- Inspect the column and counts for each model/make. Are model labels unique to each make?
- If not, create a derived column that is a unique identifier for a given plane type.

In [294]:
# 1. Drop rows with missing Make or Model
df_clean = df_filtered.dropna(subset=['Make', 'Model']).copy()
df_clean[["Make", "Model"]]




,Make,Model
16646,Mooney,M-20C
16647,Swearingen,SA-226TC
16837,Boeing,707-323B
17151,Piper,PA 16
19663,Gulfstream,AA-5B
...,...,...
88869,PIPER,PA42
88873,CIRRUS DESIGN CORP,SR22
88876,SWEARINGEN,SA226TC
88877,CESSNA,R172K


In [295]:
# 2. Clean and standardize Make and Model text formatting
df_clean['Clean.Make'] = df_clean['Make'].astype(str).str.strip().str.title()
df_clean['Clean.Model'] = df_clean['Model'].astype(str).str.strip().str.upper()

In [296]:
# Inspect the column and counts for each model/make.

df_clean[["Clean.Make", "Clean.Model"]].value_counts()

Clean.Make         Clean.Model
Cessna             172            769
Boeing             737            403
Cessna             152            314
                   182            304
                   172S           276
                                 ... 
                   P206D            1
Phantom            PHANTOM          1
James R Dernovsek  KITFOX S5        1
Orlican S R O      M-8 EAGLE        1
Piper              PA-44            1
Name: count, Length: 4349, dtype: int64

In [297]:
# Consider whether model labels are unique to each make. If not, create a derived column that is a unique identifier for a given plane type.

# 3. Create derived unique identifier for a given plane type (Make + Model combination)
df_clean['Aircraft.Type.ID'] = df_clean['Clean.Make'] + " - " + df_clean['Clean.Model']

print(f"Total rows after cleaning: {len(df_clean)}")
print(f"Unique Aircraft Types (Aircraft.Type.ID): {df_clean['Aircraft.Type.ID'].nunique()}")
print(df_clean[['Aircraft.Type.ID']].head())

Total rows after cleaning: 21397
Unique Aircraft Types (Aircraft.Type.ID): 4349
            Aircraft.Type.ID
16646         Mooney - M-20C
16647  Swearingen - SA-226TC
16837      Boeing - 707-323B
17151          Piper - PA 16
19663     Gulfstream - AA-5B


### Cleaning other columns
- there are other columns containing data that might be related to the outcome of an accident. We list a few here:
- Engine.Type
- Weather.Condition
- Number.of.Engines
- Purpose.of.flight
- Broad.phase.of.flight

Inspect and identify potential cleaning tasks in each of the above columns. Execute those cleaning tasks. 

**Note**: You do not necessarily need to impute or drop NaNs here.

In [298]:
# 1. Clean Engine.Type (convert placeholders to NaN)
def clean_engine_type(val):
    if pd.isna(val):
        return pd.NA
    s = str(val).strip().title()
    if s in ['Unk', 'Unknown', 'None', 'Lr']:
        return pd.NA
    return s

In [299]:
def clean_weather(val):
    if pd.isna(val):
        return pd.NA
    s = str(val).strip().upper()
    if s in ['UNK', 'UNK.']:
        return pd.NA
    return s

In [300]:
# 3. Clean Number.of.Engines (coerce non-numeric / negative values)
df_clean['Clean.Engines.Count'] = pd.to_numeric(df_clean['Number.of.Engines'], errors='coerce')
df_clean.loc[df_clean['Clean.Engines.Count'] < 0, 'Clean.Engines.Count'] = pd.NA

In [301]:
# 4. Clean Purpose.of.flight (convert placeholders to NaN)
def clean_purpose(val):
    if pd.isna(val):
        return pd.NA
    s = str(val).strip().title()
    if s in ['Unknown', 'Unk']:
        return pd.NA
    return s

In [302]:
# 5. Clean Broad.phase.of.flight (convert placeholders to NaN)
def clean_phase(val):
    if pd.isna(val):
        return pd.NA
    s = str(val).strip().title()
    if s in ['Unknown', 'Unk']:
        return pd.NA
    return s

In [303]:
# Apply cleaning functions
df_clean['Clean.Engine.Type'] = df_clean['Engine.Type'].apply(clean_engine_type)
df_clean['Clean.Weather'] = df_clean['Weather.Condition'].apply(clean_weather)
df_clean['Clean.Purpose'] = df_clean['Purpose.of.flight'].apply(clean_purpose)
df_clean['Clean.Phase'] = df_clean['Broad.phase.of.flight'].apply(clean_phase)

print("Cleaning successfully executed across operational and environmental columns.")

Cleaning successfully executed across operational and environmental columns.


### Column Removal
- inspect the dataframe and drop any columns that have too many NaNs

In [304]:
# Inspecting columns with NaNs
df_clean.info()

<class 'pandas.DataFrame'>
Index: 21397 entries, 16646 to 88886
Data columns (total 42 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   Event.Id                21397 non-null  str           
 1   Investigation.Type      21397 non-null  str           
 2   Accident.Number         21397 non-null  str           
 3   Event.Date              21397 non-null  datetime64[us]
 4   Location                21391 non-null  str           
 5   Country                 21396 non-null  str           
 6   Latitude                19160 non-null  str           
 7   Longitude               19154 non-null  str           
 8   Airport.Code            13959 non-null  str           
 9   Airport.Name            14043 non-null  str           
 10  Injury.Severity         20586 non-null  str           
 11  Aircraft.damage         20177 non-null  str           
 12  Aircraft.Category       21397 non-null  str           
 13

In [305]:
# Check non-null counts per column
non_null_counts = df_clean.notna().sum()
print("Non-null counts per column:")
print(non_null_counts)

# Columns with > 20,000 non-nulls
cols_to_keep = non_null_counts[non_null_counts > 20000].index.tolist()
cols_to_drop = non_null_counts[non_null_counts <= 20000].index.tolist()

print(f"\nColumns to keep (> 20,000 non-nulls): {len(cols_to_keep)}")
print(cols_to_keep)

print(f"\nColumns to drop (<= 20,000 non-nulls): {len(cols_to_drop)}")
print(cols_to_drop)

# Drop columns with <= 20,000 non-nulls
df_reduced = df_clean[cols_to_keep].copy()
print(f"\nReduced dataframe shape: {df_reduced.shape}")

Non-null counts per column:
Event.Id                  21397
Investigation.Type        21397
Accident.Number           21397
Event.Date                21397
Location                  21391
Country                   21396
Latitude                  19160
Longitude                 19154
Airport.Code              13959
Airport.Name              14043
Injury.Severity           20586
Aircraft.damage           20177
Aircraft.Category         21397
Registration.Number       21194
Make                      21397
Model                     21397
Amateur.Built             21397
Number.of.Engines         18851
Engine.Type               17454
FAR.Description           20917
Schedule                   2501
Purpose.of.flight         17727
Air.carrier               10150
Total.Fatal.Injuries      21397
Total.Serious.Injuries    21397
Total.Minor.Injuries      21397
Total.Uninjured           21397
Weather.Condition         18427
Broad.phase.of.flight      2791
Report.Status             16743
Publication.

### Save DataFrame to csv
- its generally useful to save data to file/server after its in a sufficiently cleaned or intermediate state
- the data can then be loaded directly in another notebook for further analysis
- this helps keep your notebooks and workflow readable, clean and modularized

In [306]:
df_reduced.to_csv("data/Cleaned_Aviation_Data.csv", index=False)